<a href="https://colab.research.google.com/github/lakshyacodes01/image-colorization/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Converting color image to Black and White**

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
img= tf.io.read_file('/content/Images/1000268201_693b08cb0e.jpg')

img= tf.image.decode_jpeg(img)
img=tf.cast(img,tf.float32)/255.0

In [ ]:
plt.imshow(img)

Formula to convert to grayscale from RGB

Gray= 0.299R + 0.587G + 0.114B

In [ ]:
gray_img=tf.image.rgb_to_grayscale(img)

In [ ]:
# To show converted image
plt.imshow(gray_img.numpy().squeeze(),cmap='gray')

Building the dataset pipeline

In [ ]:
pip install tensorflow-io

In [ ]:
import tensorflow_io as tfio

In [ ]:
# downloading dataset Flickr8k from kaggle
!mkdir -p ~/.kaggle
!mv kaggl.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggl.json

In [ ]:
!kaggle datasets download -d adityajn105/flickr8k

In [ ]:
!unzip flickr8k.zip

In [ ]:
import glob

In [ ]:
image_paths = glob.glob("/content/Images/*.jpg")

In [ ]:
print(len(image_paths))

In [ ]:
print(image_paths[:5])

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
# Load and preprocess image
def load_image(path):
  # read file
  img=tf.io.read_file(path)
  # decode jped -> (H,W,3)
  img= tf.image.decode_jpeg(img,channels=3)
  # resize to fix size
  img=tf.image.resize(img,(IMG_SIZE,IMG_SIZE))
  # Normalize
  img=tf.cast(img,tf.float32)/255.0
  # Convert RGB -> LAB
  lab=tfio.experimental.color.rgb_to_lab(img)
  # Split channels
  L=lab[:,:,0:1]
  ab=lab[:,:,1:]
  # Normalize LAB range
  L=L/100.0
  ab=ab/128.0
  return L,ab

In [ ]:
# Create Dataset
def create_dataset(image_paths):
  dataset=tf.data.Dataset.from_tensor_slices(image_paths)
  dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
  # apply preprocessing
  dataset=dataset.shuffle(buffer_size=1000)
  # batch
  dataset=dataset.batch(BATCH_SIZE)

  # optimize pipeline
  dataset=dataset.prefetch(tf.data.AUTOTUNE)
  return dataset

In [ ]:
import random

In [ ]:
image_paths=glob.glob("/content/Images/*.jpg")
# reduce size for faster training
images_paths=random.sample(image_paths,3000)
# create dataset
train_dataset=create_dataset(images_paths)


In [ ]:
for item in train_dataset.take(1):
  print(type(item))
  print(len(item))

Checking if dataset is correct

In [ ]:
for L, ab in train_dataset.take(1):
    L_sample = L[0]
    ab_sample = ab[0]

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(L_sample.numpy().squeeze(), cmap='gray')
plt.title("Grayscale")
plt.axis('off')
plt.show()

In [ ]:
import numpy as np
import cv2

L = L_sample.numpy() * 100
ab = ab_sample.numpy() * 128

lab = np.concatenate([L, ab], axis=-1).astype(np.float32)
rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

plt.imshow(rgb)
plt.title("Original Color")
plt.axis('off')
plt.show()

**BUILDING THE MODEL**

In [ ]:
from tensorflow.keras import Sequential,layers,Model


In [ ]:
# Loading ResNet50 model
base_model=tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE,IMG_SIZE,3)
)

In [ ]:
for layer in base_model.layers:
  layer.trainable=False

In [ ]:
inputs=tf.keras.Input(shape=(224,224,1))

In [ ]:
x=layers.Concatenate()([inputs,inputs,inputs])

In [ ]:
x=base_model(x)

In [ ]:
x=layers.Conv2DTranspose(256,3,strides=2,padding='same',activation='relu')(x)
x=layers.Conv2DTranspose(128,3,strides=2,padding='same',activation='relu')(x)
x=layers.Conv2DTranspose(64,3,strides=2,padding='same',activation='relu')(x)
x=layers.Conv2DTranspose(32,3,strides=2,padding='same',activation='relu')(x)
x=layers.Conv2DTranspose(16,3,strides=2,padding='same',activation='relu')(x)

outputs=layers.Conv2D(2,3,padding='same',activation='tanh')(x)




In [ ]:
model=Model(inputs,outputs)

In [ ]:
model.summary()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_paths,val_paths=train_test_split(image_paths,test_size=0.2,random_state=42 )

In [ ]:
train_dataset = create_dataset(train_paths)
val_dataset = create_dataset(val_paths)

In [ ]:
model.compile(optimizer='adam',loss='mse',metrics=['accuracy'])

In [ ]:
history=model.fit(train_dataset,validation_data=val_dataset,epochs=10)

In [ ]:
import tensorflow as tf
import tensorflow_io as tfio
import numpy as np
import cv2
import matplotlib.pyplot as plt

IMG_SIZE = 224

# load image
img = tf.io.read_file('/content/Images/1012212859_01547e3f17.jpg')
img = tf.image.decode_jpeg(img, channels=3)
img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
img = tf.cast(img, tf.float32) / 255.0

# convert to LAB
lab = tfio.experimental.color.rgb_to_lab(img)

L = lab[:, :, 0:1]
ab = lab[:, :, 1:]

# normalize
L_input = L / 100.0

In [ ]:
L_input = tf.expand_dims(L_input, axis=0)  # add batch

pred_ab = model.predict(L_input)

In [ ]:
# remove batch
pred_ab = pred_ab[0]

# denormalize
L = L.numpy()
pred_ab = pred_ab * 128

# combine
lab = np.concatenate([L, pred_ab], axis=-1).astype(np.float32)

# LAB → RGB
rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

In [ ]:
plt.figure(figsize=(12,4))

# grayscale input
plt.subplot(1,3,1)
plt.imshow(L.squeeze(), cmap='gray')
plt.title("Input (Grayscale)")
plt.axis('off')

# predicted color
plt.subplot(1,3,2)
plt.imshow(rgb)
plt.title("Predicted Color")
plt.axis('off')

# original (for comparison)
plt.subplot(1,3,3)
plt.imshow(img)
plt.title("Original Image")
plt.axis('off')

plt.show()

Changing Loss Function to MAE

In [ ]:
model.compile(optimizer='adam',loss='mae',metrics=['accuracy'])

In [ ]:
history=model.fit(train_dataset,validation_data=val_dataset,epochs=30)

NOW MAKING U-NET Architecture

In [ ]:
# Loading ResNet50 model
base_model=tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE,IMG_SIZE,3)
)

In [ ]:
for layer in base_model.layers:
  layer.trainable=False

In [ ]:
base_model.summary()

In [ ]:
layer_names=[
    "conv1_relu",
    "conv2_block3_out",
    "conv3_block4_out",
    "conv4_block6_out",
    "conv5_block3_out"
]

In [ ]:
layers_outputs=[base_model.get_layer(name).output for name in layer_names]

In [ ]:
encoder= Model(inputs=base_model.input,outputs=layers_outputs)

In [ ]:
inputs =tf.keras.Input(shape=(224,224,1))

x=layers.Concatenate()([inputs,inputs,inputs])

In [ ]:
f1,f2,f3,f4,f5=encoder(x)

In [ ]:
# 7 → 14
x = layers.Conv2DTranspose(512, 3, strides=2, padding='same', activation='relu')(f5)
x = layers.Concatenate()([x, f4])

# 14 → 28
x = layers.Conv2DTranspose(256, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Concatenate()([x, f3])

# 28 → 56
x = layers.Conv2DTranspose(128, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Concatenate()([x, f2])

# 56 → 112
x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Concatenate()([x, f1])

# 112 → 224
x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)

In [ ]:
outputs = layers.Conv2D(2, 3, padding='same', activation='tanh')(x)

In [ ]:
model = Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer='adam',
    loss='mae',
    metrics=['mae']
)

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=150,   # high number is fine
    callbacks=[early_stop]
)